# SOSD.jl on Colab: GPU smoke test, tests and benchmarks

Runs the `gpu` branch of [SOSD.jl](https://github.com/bachrathyd/SOSD.jl) on a Colab GPU and writes every result to Google Drive.

**Before you start:** Runtime → Change runtime type → Python 3 + a GPU (Julia is driven from the Python runtime, because only Python can mount Google Drive).

Steps: 1 settings · 2 GPU + Drive · 3 Julia · 4 CUDA smoke test · 5 code · 6 tests · 7 benchmark

When you are done: Runtime → Disconnect and delete runtime, otherwise an idle GPU keeps consuming compute units.

## 1. Settings

In [ ]:
REPO = "https://github.com/bachrathyd/SOSD.jl"
BRANCH = "gpu"
DRIVE_FOLDER = "Colab Notebooks/SOSD_GPU"  # folder in My Drive that receives the results (created if missing)
JULIA_CHANNEL = "1.12"
BENCH_MODE = "full"   # "quick" (~2 min) or "full"

## 2. GPU and Google Drive

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

In [ ]:
import os, datetime, subprocess
from google.colab import drive
drive.mount('/content/drive')

root = '/content/drive/MyDrive'
OUT = os.path.join(root, DRIVE_FOLDER)
os.makedirs(OUT, exist_ok=True)

gpu = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
                     capture_output=True, text=True).stdout.strip().split('\n')[0]
tag = (gpu or 'noGPU').replace(' ', '_')
RUN = os.path.join(OUT, 'runs', datetime.datetime.now().strftime('%Y%m%d_%H%M') + '_' + tag)
os.makedirs(RUN, exist_ok=True)
print('GPU :', gpu or 'NONE -- select a GPU runtime!')
print('results ->', RUN)

## 3. Julia (juliaup, ~1 min)

In [ ]:
os.environ['PATH'] = '/root/.juliaup/bin:' + os.environ['PATH']
os.environ['JULIA_NUM_THREADS'] = 'auto'
if not os.path.exists('/root/.juliaup/bin/julia'):
    !curl -fsSL https://install.julialang.org | sh -s -- --yes --default-channel {JULIA_CHANNEL} > /dev/null
!julia --version
!nproc

## 4. CUDA.jl smoke test (first run ~5–10 min: package install + CUDA runtime download)

Checks that CUDA.jl and KernelAbstractions work, runs a tiny kernel, and measures the GEMM throughput in Float32 and Float64 (the FP64/FP32 ratio decides whether Float64 sweeps are worth it on this GPU).

In [ ]:
%%writefile /content/cuda_smoke.jl
using Pkg
Pkg.activate("/content/cuda_env")
Pkg.add(["CUDA", "KernelAbstractions"]; io=devnull)
using CUDA, KernelAbstractions, LinearAlgebra

CUDA.versioninfo()
println("functional: ", CUDA.functional(), "   device: ", CUDA.name(CUDA.device()))

@kernel function axpy_kernel!(y, a, x)
    i = @index(Global)
    @inbounds y[i] += a * x[i]
end
x = CUDA.rand(Float64, 10^7); y = CUDA.zeros(Float64, 10^7)
axpy_kernel!(CUDABackend())(y, 2.0, x; ndrange=length(y))
KernelAbstractions.synchronize(CUDABackend())
println("KernelAbstractions axpy correct: ", Array(y) ≈ 2 .* Array(x))

n = 4096
for T in (Float32, Float64)
    A = CUDA.rand(T, n, n); B = CUDA.rand(T, n, n); C = similar(A)
    mul!(C, A, B); CUDA.synchronize()
    t = CUDA.@elapsed for _ in 1:5; mul!(C, A, B); end
    println(rpad(string(T), 8), " GEMM: ", round(5 * 2 * n^3 / t / 1e12, digits=2), " TFLOPS")
end

In [ ]:
!julia /content/cuda_smoke.jl 2>&1 | tee "{RUN}/cuda_smoke.log"

## 5. Code (clone or update the `gpu` branch)

In [ ]:
REPO_DIR = '/content/SOSD.jl'
has_branch = subprocess.run(['git', 'ls-remote', '--exit-code', '--heads', REPO, BRANCH],
                            capture_output=True).returncode == 0
if not has_branch:
    print(f'Branch "{BRANCH}" is not on GitHub yet -- steps 5-7 will work once it is pushed.')
elif os.path.isdir(REPO_DIR):
    !cd {REPO_DIR} && git fetch -q --depth 1 origin {BRANCH} && git reset -q --hard FETCH_HEAD
else:
    !git clone -q -b {BRANCH} --depth 1 {REPO} {REPO_DIR}
if has_branch:
    !cd {REPO_DIR} && git log -1 --format='%h %ad %s' --date=short

## 6. GPU tests (CPU reference vs GPU, ρ relative difference < 1e-8 in Float64)

First run on a fresh runtime: ~10 min (package install + precompilation).

In [ ]:
if has_branch:
    !cd {REPO_DIR} && julia --project=gpu -e 'using Pkg; Pkg.instantiate(); Pkg.precompile()' 2>&1 | tail -3
    !cd {REPO_DIR} && julia --project=gpu gpu/test_gpu.jl 2>&1 | tee "{RUN}/test_gpu.log"

## 7. Benchmark (time per ρ and per stability chart)

CPU timings use the Colab VM's cores (`nproc` above), not a 16-core workstation; the CSV records the thread count.

In [ ]:
if has_branch:
    mode = 'quick' if BENCH_MODE == 'quick' else ''
    !cd {REPO_DIR} && julia --project=gpu gpu/bench_gpu.jl "{RUN}" {mode} 2>&1 | tee "{RUN}/bench_gpu.log"

## 8. Stability chart from the benchmark run

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
g = pd.read_csv(f'{RUN}/chart_grid.csv'); b = pd.read_csv(f'{RUN}/chart_boundary.csv')
n = np.sort(g.n_rpm.unique()); w = np.sort(g.w_m.unique())
R = g.pivot(index='w_m', columns='n_rpm', values='rho').loc[w, n].values
fig, ax = plt.subplots(figsize=(9, 4.5))
pc = ax.pcolormesh(n, w * 1e3, np.log10(R), cmap='RdBu_r', vmin=-1, vmax=1, shading='auto')
ax.contour(n, w * 1e3, R, levels=[1.0], colors='k', linewidths=1)
ax.plot(b.n_rpm, b.w_lim_m * 1e3, 'y.', ms=3, label='multisection boundary')
fig.colorbar(pc, label=r'$\log_{10}\rho$')
ax.set_xlabel(r'spindle speed $n$ [rpm]'); ax.set_ylabel(r'depth of cut $w$ [mm]'); ax.legend(loc='upper left')
fig.savefig(f'{RUN}/chart_milling2dof.png', dpi=150, bbox_inches='tight'); plt.show()
print(pd.read_csv(f'{RUN}/bench_results.csv').to_string())